# Brain Age Prediction: Training T1 & Ensemble con FLAIR
Questo notebook è stato aggiornato per:
1. Permettere la scelta della modalità MRI (FLAIR o T1w).
2. Addestrare il modello SFCN sulle immagini T1 (sfruttando le stesse tecniche di bilanciamento e 5-Fold CV usate per il FLAIR).
3. Eseguire un **Ensemble** finale: caricare i pesi del modello T1 (appena addestrato) e i pesi del modello FLAIR (che devi aver pre-caricato), facendo una media delle probabilità per ottenere una predizione combinata estremamente robusta sul Test Set.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('/kaggle/working/SFCN')

In [ ]:
import os
import json
import glob
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold, train_test_split

# Import dal tuo repository
from dp_model.model_files.sfcn import SFCN
from dp_model import dp_utils as dpu
from train import train_model

# Creiamo la directory globale per salvare tutti i grafici su Kaggle
PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Dataset Custom (con selezione Modalità T1/FLAIR)

In [ ]:
class BrainAgeDataset(Dataset):
    def __init__(self, data_dir, modality='T1w', is_train=True):
        self.data_dir = data_dir
        self.modality = modality # 'FLAIR' o 'T1w' (o 'T1')
        self.is_train = is_train
        self.subject_dirs = sorted(glob.glob(os.path.join(data_dir, "sub-*")))
        self.samples = []
        
        self.bin_range = [0, 70]
        self.bin_step = 1
        self.sigma = 1.0
        
        for subj_dir in self.subject_dirs:
            subj_id = os.path.basename(subj_dir)
            
            # Cerca il file NIfTI corrispondente alla modalità scelta
            # Modifica il nome del file qui sotto se i tuoi T1 si chiamano diversamente (es. T1 anzichè T1w)
            nii_path = os.path.join(subj_dir, f"{subj_id}_{self.modality}_MNI152_1mm.nii")
            
            if not os.path.exists(nii_path):
                nii_path = nii_path + ".gz"
                if not os.path.exists(nii_path):
                    # Fallback nel caso si chiami T1 anzichè T1w
                    if self.modality == 'T1w':
                        nii_path_alt = os.path.join(subj_dir, f"{subj_id}_T1_MNI152_1mm.nii.gz")
                        if os.path.exists(nii_path_alt):
                            nii_path = nii_path_alt
                        else:
                            print(f"Saltato {subj_id}: NIfTI ({self.modality}) non trovato.")
                            continue
                    else:
                        print(f"Saltato {subj_id}: NIfTI ({self.modality}) non trovato.")
                        continue
                    
            json_path = os.path.join(subj_dir, f"{subj_id}_participant_info.json")
            if not os.path.exists(json_path):
                print(f"Saltato {subj_id}: JSON non trovato in {json_path}")
                continue
                
            with open(json_path, 'r') as f:
                info = json.load(f)
                
            participant_info = info.get("participant_info", {})
            age_cat_val = participant_info.get("age_scan")
            
            if age_cat_val is None:
                continue
            
            try:
                age_cat = int(age_cat_val) - 1
                true_age = 3 + age_cat * 5
                y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
            except Exception as e:
                continue
            
            self.samples.append({
                "nii_path": nii_path,
                "label_vect": y,
                "true_age": true_age
            })

        if len(self.samples) == 0:
            print(f"ATTENZIONE: Nessun campione trovato per la modalità {self.modality}.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        if self.is_train:
            if np.random.rand() > 0.5:
                data = np.flip(data, axis=0).copy()
                
            dx = np.random.randint(-2, 3)
            dy = np.random.randint(-2, 3)
            dz = np.random.randint(-2, 3)
        else:
            dx, dy, dz = 0, 0, 0
            
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_data, label_vect, sample['true_age']

## 2. Addestramento del Modello T1

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/elenaschgor/dataset-2-t1-flair/ds004199_final/"
CURRENT_MODALITY = 'T1w'  # <-- Modificato a T1w per addestrare sulle T1

full_train_dataset = BrainAgeDataset(KAGGLE_DATA_DIR, modality=CURRENT_MODALITY, is_train=True)
full_val_dataset   = BrainAgeDataset(KAGGLE_DATA_DIR, modality=CURRENT_MODALITY, is_train=False) 
dataset_size = len(full_train_dataset)

print(f"Trovati {dataset_size} campioni validi per {CURRENT_MODALITY}.\n")

if dataset_size > 0:
    all_ages = [sample['true_age'] for sample in full_train_dataset.samples]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Split fisso (random_state=42 garantisce di usare gli STESSI pazienti del test set del modello FLAIR)
    all_indices = np.arange(dataset_size)
    train_val_idx, test_idx = train_test_split(
        all_indices, 
        test_size=0.10, 
        random_state=42, 
        stratify=all_ages
    )
    
    # --- ADDESTRAMENTO 5-FOLD --- 
    k_folds = 5
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    fold_results = []
    best_overall_mae = float('inf')
    best_overall_model_state = None
    train_val_ages = np.array(all_ages)[train_val_idx]
    
    for fold, (fold_train_idx, fold_val_idx) in enumerate(skf.split(train_val_idx, train_val_ages)):
        print(f"\n==============================================")
        print(f"      ADDESTRAMENTO {CURRENT_MODALITY} - FOLD {fold + 1}/{k_folds}")
        print(f"==============================================")
        
        train_idx = train_val_idx[fold_train_idx]
        val_idx = train_val_idx[fold_val_idx]
        
        train_dataset = torch.utils.data.Subset(full_train_dataset, train_idx)
        val_dataset = torch.utils.data.Subset(full_val_dataset, val_idx)
        
        # Weighted Sampler per bilanciare le classi
        fold_train_ages = [full_train_dataset.samples[i]['true_age'] for i in train_idx]
        unique_ages, counts = np.unique(fold_train_ages, return_counts=True)
        age_weight_dict = {age: 1.0 / count for age, count in zip(unique_ages, counts)}
        sample_weights = [age_weight_dict[age] for age in fold_train_ages]
        sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
        
        train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, num_workers=2)
        val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
        
        model = SFCN(output_dim=70)
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        model = model.to(device)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=0.001)
        
        trained_model, train_losses, val_losses, val_maes = train_model(
            model=model, 
            train_loader=train_loader, 
            val_loader=val_loader, 
            optimizer=optimizer, 
            device=device, 
            epochs=500,
            step_size=200,
            gamma=0.3,
            patience=100,
            fold_idx=fold+1
        )
        
        best_fold_mae = min(val_maes)
        fold_results.append(best_fold_mae)
        
        if best_fold_mae < best_overall_mae:
            best_overall_mae = best_fold_mae
            best_overall_model_state = trained_model.state_dict().copy()
            
    # Salvataggio del modello T1
    model_save_path_T1 = f"/kaggle/working/sfcn_best_model_{CURRENT_MODALITY}.pth"
    model_T1_clean = SFCN(output_dim=70)
    if torch.cuda.device_count() > 1:
        # Estraiamo dal DataParallel per salvarlo pulito
        torch.save({k.replace('module.', ''): v for k, v in best_overall_model_state.items()}, model_save_path_T1)
    else:
        torch.save(best_overall_model_state, model_save_path_T1)
        
    print(f"\n[!] Pesi del modello T1 salvati in: {model_save_path_T1}")

## 3. L'Ensemble Definitivo (FLAIR + T1) sul Test Set Incontaminato

In [ ]:
# PERCORSO DEL TUO MODELLO FLAIR (modifica se necessario)
# Assicurati di aver fatto l'upload del file .pth del FLAIR in Kaggle
# Esempio: "/kaggle/input/flair-model/sfcn_best_model_fold_CV.pth"
PATH_MODELLO_FLAIR = "/kaggle/input/sfcn-flair-weights/sfcn_best_model_fold_CV.pth"
PATH_MODELLO_T1 = model_save_path_T1

if not os.path.exists(PATH_MODELLO_FLAIR):
    print(f"\n[ATTENZIONE] Il modello FLAIR non è stato trovato in {PATH_MODELLO_FLAIR}.")
    print("Carica il file .pth del FLAIR su Kaggle e aggiorna il PATH_MODELLO_FLAIR per avviare l'ensemble.")
else:
    print("\n==============================================")
    print("      AVVIO ENSEMBLE (T1 + FLAIR)")
    print("==============================================")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Caricamento Modello FLAIR
    model_flair = SFCN(output_dim=70)
    model_flair.load_state_dict(torch.load(PATH_MODELLO_FLAIR, map_location=device))
    model_flair.to(device)
    model_flair.eval()
    
    # 2. Caricamento Modello T1
    model_t1 = SFCN(output_dim=70)
    model_t1.load_state_dict(torch.load(PATH_MODELLO_T1, map_location=device))
    model_t1.to(device)
    model_t1.eval()
    
    # Preparazione dei due DataLoader (stessi identici pazienti grazie al random_state=42!)
    # Test set T1
    test_dataset_t1 = torch.utils.data.Subset(full_val_dataset, test_idx)
    test_loader_t1 = DataLoader(test_dataset_t1, batch_size=1, shuffle=False)
    
    # Test set FLAIR
    full_val_dataset_flair = BrainAgeDataset(KAGGLE_DATA_DIR, modality='FLAIR', is_train=False)
    test_dataset_flair = torch.utils.data.Subset(full_val_dataset_flair, test_idx)
    test_loader_flair = DataLoader(test_dataset_flair, batch_size=1, shuffle=False)
    
    ensemble_errors = []
    all_true_ages = []
    all_ensemble_preds = []
    bin_centers = np.arange(0, 70, 1)
    
    with torch.no_grad():
        # Iteriamo simultaneamente su entrambe le modalità
        for (inputs_t1, _, true_age), (inputs_flair, _, _) in zip(test_loader_t1, test_loader_flair):
            inputs_t1 = inputs_t1.to(device)
            inputs_flair = inputs_flair.to(device)
            true_age_val = true_age.item()
            
            # Predizione T1
            out_t1 = model_t1(inputs_t1)[0].view(1, -1)
            prob_t1 = torch.exp(out_t1).cpu().numpy()
            
            # Predizione FLAIR
            out_flair = model_flair(inputs_flair)[0].view(1, -1)
            prob_flair = torch.exp(out_flair).cpu().numpy()
            
            # ENSEMBLE: Media delle probabilità gaussiane!
            ensemble_prob = (prob_t1 + prob_flair) / 2.0
            
            # Età predetta combinata
            ensemble_pred_age = ensemble_prob @ bin_centers
            ensemble_pred_age = ensemble_pred_age[0]
            
            error = abs(ensemble_pred_age - true_age_val)
            ensemble_errors.append(error)
            
            all_true_ages.append(true_age_val)
            all_ensemble_preds.append(ensemble_pred_age)
            
    ensemble_mae = np.mean(ensemble_errors)
    print(f"\n>>> MAE ENSEMBLE (T1 + FLAIR) SUL TEST SET CATTIVO: {ensemble_mae:.2f} anni <<<")
    
    # PLOT DELL'ENSEMBLE
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.scatter(all_true_ages, all_ensemble_preds, color='purple', edgecolor='k', s=80, alpha=0.8)
    min_val = min(min(all_true_ages), min(all_ensemble_preds)) - 2
    max_val = max(max(all_true_ages), max(all_ensemble_preds)) + 2
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Predizione Perfetta')
    plt.title('Ensemble (T1+FLAIR): Età Reale vs Predetta')
    plt.xlabel('Età Reale (Anni)')
    plt.ylabel('Età Predetta (Anni)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.subplot(1, 2, 2)
    plt.bar(range(1, len(ensemble_errors) + 1), ensemble_errors, color='gold', edgecolor='k', alpha=0.8)
    plt.axhline(y=ensemble_mae, color='k', linestyle='dashed', linewidth=2, label=f'MAE Medio: {ensemble_mae:.2f} anni')
    plt.title('Errore Assoluto Ensemble per Paziente')
    plt.xlabel('Indice Paziente')
    plt.ylabel('Errore Assoluto (Anni Sbagliati)')
    plt.xticks(range(1, len(ensemble_errors) + 1))
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, '03_ensemble_test_evaluation.png'))
    plt.show()
